In [ ]:
import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, SEASONS
sys.path.insert(1, f"{REPO_PATH}")

# Helper functions

In [25]:
from typing import List
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

def encode_and_onehot_transform_teams(
    season_dfs: List[pd.DataFrame],
    home_col: str = "home",
    away_col: str = "away",
    date_col: str = "date"
) -> tuple[pd.DataFrame, ColumnTransformer]:
    """
    Combines the processing and one-hot encoding of team-related features across multiple seasons.

    Args:
        season_dfs (List[pd.DataFrame]): List of seasonal match DataFrames.
        home_col (str): Column name for home team.
        away_col (str): Column name for away team.
        date_col (str): Column name for match date.

    Returns:
        Tuple:
            - pd.DataFrame: Combined and transformed DataFrame with one-hot encoded features.
            - ColumnTransformer: Fitted transformer for reuse.
    """
    known_teams = set()
    processed_dfs = []

    for season_idx, df in enumerate(season_dfs):
        df = df.copy()
        df["season"] = season_idx + 1  # Season index

        # Team-season IDs
        df["home_team_season"] = df[home_col] + "_" + df["season"].astype(str)
        df["away_team_season"] = df[away_col] + "_" + df["season"].astype(str)

        # Symmetric team matchup ID
        df["team_matchup"] = df[[home_col, away_col]].apply(
            lambda row: "_".join(sorted([row[home_col], row[away_col]])), axis=1
        )

        # New team flags
        current_teams = set(df[home_col]).union(set(df[away_col]))

        if season_idx == 0:
            df["is_new_home_team"] = False
            df["is_new_away_team"] = False
        else:
            df["is_new_home_team"] = ~df[home_col].isin(known_teams)
            df["is_new_away_team"] = ~df[away_col].isin(known_teams)

        known_teams.update(current_teams)
        processed_dfs.append(df)

    # Combine all seasons
    combined_df = pd.concat(processed_dfs, ignore_index=True)

    # Categorical features to one-hot encode
    categorical_features = [
        home_col,
        away_col,
        "home_team_season",
        "away_team_season",
        "team_matchup"
    ]

    transformer = ColumnTransformer(
        transformers=[
            ("team_onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
        ],
    )

    transformed_array = transformer.fit_transform(combined_df)
    transformed_feature_names = transformer.get_feature_names_out()
    transformed_df = pd.DataFrame(transformed_array, columns=transformed_feature_names)

    return transformed_df, transformer




# Read Processed Data

In [4]:
processed_data_path='../../data/processed/premier_league/'

In [8]:
seasons=sorted(SEASONS)

In [9]:
data_dfs=[pd.read_csv(f"{processed_data_path}/{season}/all_data_df.csv") for season in seasons]

In [26]:
teams_encoding, transformer=encode_and_onehot_transform_teams(data_dfs)

In [27]:
teams_encoding.columns

Index(['team_onehot__home_Arsenal', 'team_onehot__home_Aston Villa',
       'team_onehot__home_Bournemouth', 'team_onehot__home_Brentford',
       'team_onehot__home_Brighton', 'team_onehot__home_Burnley',
       'team_onehot__home_Chelsea', 'team_onehot__home_Crystal Palace',
       'team_onehot__home_Everton', 'team_onehot__home_Fulham',
       ...
       'team_onehot__team_matchup_Southampton_Wolves',
       'team_onehot__team_matchup_Tottenham_Watford',
       'team_onehot__team_matchup_Tottenham_West Brom',
       'team_onehot__team_matchup_Tottenham_West Ham',
       'team_onehot__team_matchup_Tottenham_Wolves',
       'team_onehot__team_matchup_Watford_West Ham',
       'team_onehot__team_matchup_Watford_Wolves',
       'team_onehot__team_matchup_West Brom_West Ham',
       'team_onehot__team_matchup_West Brom_Wolves',
       'team_onehot__team_matchup_West Ham_Wolves'],
      dtype='object', length=579)